# Question 2: News Topic Clustering and Semantic Embedding Comparison

**Course practical:** NLP / Machine Learning exam  
**Task:** Automatically organize news articles into meaningful topics without predefined labels.  
**Dataset:** Kaggle News Category Dataset by Rishabh Misra.

This notebook is structured exactly around the Question 2 marking guide. It includes the implementation, explanations, tables, and visuals needed for presentation.

## Rubric Map

| Exam part | What this notebook shows |
| --- | --- |
| (a) Data preparation | Loads the dataset, displays first 10 records, explains columns, computes document statistics, identifies noise. |
| (b) Preprocessing | Cleans text with regex, lowercases, tokenizes, removes stopwords, normalizes word forms, shows before/after examples. |
| (c) NLP exploration | Displays top words, bigrams, and trigrams with charts and explanation. |
| (d) Representation | Builds Bag-of-Words and TF-IDF matrices, compares weighting and sparsity. |
| (e) Similarity | Uses cosine similarity on TF-IDF vectors and displays most similar article pairs. |
| (f) Clustering | Applies K-Means, uses elbow/silhouette selection, displays cluster samples and topic labels. |
| (g) Transformers | Generates SentenceTransformer embeddings and clusters them with K-Means. |
| (h) Evaluation | Compares TF-IDF and transformer clustering with keywords, labels, examples, and silhouette scores. |

## 1. Setup

The companion script `q2_news_clustering.py` generated the final outputs used in this notebook. The notebook also includes runnable code so the logic can be inspected and repeated.

In [ ]:
import re
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import Image, display
from nltk import ngrams
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity

BASE_DIR = Path.cwd()
if not (BASE_DIR / 'News_Category_Dataset_v3.json').exists() and (BASE_DIR / 'News_Category').exists():
    BASE_DIR = BASE_DIR / 'News_Category'

DATA_FILE = BASE_DIR / 'News_Category_Dataset_v3.json'
OUTPUT_DIR = BASE_DIR / 'outputs'
RANDOM_STATE = 42

pd.set_option('display.max_colwidth', 140)
plt.style.use('seaborn-v0_8-whitegrid')
print('Dataset:', DATA_FILE)
print('Outputs:', OUTPUT_DIR)

## (a) Data Preparation and Understanding

The dataset contains news article metadata and text. For topic analysis, the most important columns are:

- `headline`: short, high-signal summary of the article topic.
- `short_description`: adds more context to the headline.
- `category`: original editorial category. It is not used as a label during clustering, but it helps validate whether clusters make sense.
- `date`, `authors`, and `link`: useful metadata, but less central to text clustering.

In [ ]:
def repair_mojibake(value):
    """Repair common UTF-8 text accidentally decoded as Latin-1."""
    if not isinstance(value, str) or '?' not in value:
        return value
    try:
        return value.encode('latin-1').decode('utf-8')
    except UnicodeError:
        return value

raw_df = pd.read_json(DATA_FILE, lines=True)
raw_df['headline'] = raw_df['headline'].fillna('').apply(repair_mojibake)
raw_df['short_description'] = raw_df['short_description'].fillna('').apply(repair_mojibake)
raw_df['document'] = (raw_df['headline'] + ' ' + raw_df['short_description']).str.strip()
raw_df = raw_df[raw_df['document'].str.len() > 0].copy()

raw_df[['headline', 'category', 'short_description', 'authors', 'date']].head(10)

In [ ]:
# Use the final sampled output from the script so notebook results match the submitted report.
clustered_df = pd.read_csv(OUTPUT_DIR / 'q2_clustered_news_sample.csv')
tfidf_summary = pd.read_csv(OUTPUT_DIR / 'q2_tfidf_cluster_summary.csv')
embedding_summary = pd.read_csv(OUTPUT_DIR / 'q2_embedding_cluster_summary.csv')
similarity_pairs = pd.read_csv(OUTPUT_DIR / 'q2_similarity_pairs.csv')
elbow_scores = pd.read_csv(OUTPUT_DIR / 'q2_elbow_scores.csv')
model_comparison = pd.read_csv(OUTPUT_DIR / 'q2_model_comparison.csv')

clean_tokens = clustered_df['clean_text'].fillna('').astype(str).str.split().tolist()
statistics = pd.DataFrame({
    'Statistic': ['Full dataset documents', 'Documents used in final run', 'Average cleaned document length', 'Vocabulary size after tokenization'],
    'Value': [
        len(raw_df),
        len(clustered_df),
        round(np.mean([len(tokens) for tokens in clean_tokens]), 2),
        len(set(token for tokens in clean_tokens for token in tokens))
    ]
})
statistics

### Noise and Inconsistencies

The raw news text contains several issues that can reduce clustering quality:

1. **Punctuation and symbols:** headlines contain parentheses, apostrophes, commas, and other symbols that create extra token variants.
2. **Capitalization differences:** words like `Trump`, `TRUMP`, and `trump` should be treated as the same token.
3. **Short or empty descriptions:** articles with little text produce weak vectors and may be pulled into broad clusters.
4. **Named entities and source style language:** names, dates, and repeated media wording can dominate clusters if not normalized.
5. **Encoding artifacts:** characters such as `???` sometimes appear when text encoding is misread; these should be repaired.

The category chart below is not used for training; it is shown only to understand the sampled data composition.

![Top original news categories](outputs/q2_category_distribution.png)

## (b) Text Preprocessing and Normalization

The cleaning stage prepares text for vectorization and clustering. It lowercases text, removes URLs/punctuation/special characters, tokenizes words, removes stopwords, and applies a small lemmatization fallback.

In [ ]:
EXTRA_STOPWORDS = {
    'said', 'say', 'says', 'new', 'year', 'years', 'day', 'week', 'time',
    'people', 'just', 'like', 'huffpost'
}
STOPWORDS = set(ENGLISH_STOP_WORDS).union(EXTRA_STOPWORDS)


def simple_lemma(token):
    if len(token) > 4 and token.endswith('ies'):
        return token[:-3] + 'y'
    if len(token) > 5 and token.endswith('ing'):
        return token[:-3]
    if len(token) > 4 and token.endswith('ed'):
        return token[:-2]
    if len(token) > 3 and token.endswith('s'):
        return token[:-1]
    return token


def clean_and_tokenize(text):
    text = str(text).lower()
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = re.findall(r'[a-z]+', text)
    return [simple_lemma(token) for token in tokens if token not in STOPWORDS and len(token) > 2]

preprocess_examples = raw_df[['document']].head(4).copy()
preprocess_examples['tokens'] = preprocess_examples['document'].apply(clean_and_tokenize)
preprocess_examples['cleaned_text'] = preprocess_examples['tokens'].apply(lambda row: ' '.join(row))
preprocess_examples.rename(columns={'document': 'original_text'})

**Why preprocessing helps clustering:** after cleaning, articles about the same topic share more comparable vocabulary. For example, punctuation and capitalization no longer split the same word into different forms, and stopword removal leaves more topic-bearing terms.

## (c) NLP Exploration and Pattern Discovery

Frequent terms give a quick view of dominant topics. N-grams are even more useful because many news topics appear as phrases, such as `donald trump`, `white house`, `health care`, and `climate change`.

In [ ]:
word_counts = Counter(token for tokens in clean_tokens for token in tokens).most_common(10)
bigram_counts = Counter(' '.join(tokens[i:i+2]) for tokens in clean_tokens for i in range(len(tokens)-1)).most_common(10)
trigram_counts = Counter(' '.join(tokens[i:i+3]) for tokens in clean_tokens for i in range(len(tokens)-2)).most_common(10)

display(pd.DataFrame(word_counts, columns=['term', 'count']))
display(pd.DataFrame(bigram_counts, columns=['bigram', 'count']))
display(pd.DataFrame(trigram_counts, columns=['trigram', 'count']))

### Visual Exploration

![Top words](outputs/q2_top_words.png)

![Top bigrams](outputs/q2_top_bigrams.png)

![Top trigrams](outputs/q2_top_trigrams.png)

**Interpretation:** The most common words and phrases point to repeated themes in the sample. Political terms such as `donald trump`, public-policy phrases such as `health care`, and social topics such as `black live matter` help explain why clusters later form around politics, health/social issues, lifestyle, entertainment, and family-related content.

## (d) Text Representation

Two traditional representations are created:

- **Bag-of-Words:** stores raw term counts.
- **TF-IDF:** stores weighted scores that reduce the importance of very common terms and increase the importance of distinctive terms.

TF-IDF is usually better for similarity and clustering because a rare, topic-specific term should matter more than a general repeated word.

In [ ]:
bow_vectorizer = CountVectorizer(max_features=5000, min_df=3, max_df=0.85)
bow_matrix = bow_vectorizer.fit_transform(clustered_df['clean_text'].fillna(''))

tfidf_vectorizer = TfidfVectorizer(max_features=5000, min_df=3, max_df=0.85)
tfidf_matrix = tfidf_vectorizer.fit_transform(clustered_df['clean_text'].fillna(''))

representation_summary = pd.DataFrame({
    'Representation': ['Bag-of-Words', 'TF-IDF'],
    'Documents': [bow_matrix.shape[0], tfidf_matrix.shape[0]],
    'Features': [bow_matrix.shape[1], tfidf_matrix.shape[1]],
    'Non-zero values': [bow_matrix.nnz, tfidf_matrix.nnz],
    'Sparsity (%)': [
        round(100 * (1 - bow_matrix.nnz / (bow_matrix.shape[0] * bow_matrix.shape[1])), 2),
        round(100 * (1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])), 2)
    ]
})
representation_summary

In [ ]:
# Example TF-IDF weights for one article.
doc_index = 0
feature_names = np.array(tfidf_vectorizer.get_feature_names_out())
row = tfidf_matrix[doc_index]
nonzero = row.nonzero()[1]
weights = row.data
example_weights = pd.DataFrame({'term': feature_names[nonzero], 'tfidf_score': weights})
example_weights = example_weights.sort_values('tfidf_score', ascending=False).head(12)
print(clustered_df.loc[doc_index, 'headline'])
example_weights

## (e) Document Similarity Analysis

Cosine similarity measures whether two TF-IDF vectors point in a similar direction. It is useful for finding articles that discuss related ideas, even if their total lengths differ.

In [ ]:
def shared_terms_for_pair(row):
    a_terms = set(clustered_df.loc[int(row['doc_a']), 'clean_text'].split())
    b_terms = set(clustered_df.loc[int(row['doc_b']), 'clean_text'].split())
    return ', '.join(sorted(a_terms.intersection(b_terms))[:12])

similarity_display = similarity_pairs.copy()
similarity_display['shared_terms'] = similarity_display.apply(shared_terms_for_pair, axis=1)
similarity_display[['doc_a', 'doc_b', 'score', 'headline_a', 'headline_b', 'category_a', 'category_b', 'shared_terms']]

![Top cosine similarity scores](outputs/q2_similarity_scores.png)

**Interpretation:** The highest-ranked pairs tend to share important names, topic phrases, or category-specific vocabulary. This demonstrates why TF-IDF cosine similarity can identify related articles without using labels.

## (f) Machine Learning: K-Means Clustering

K-Means is trained on the TF-IDF matrix. The number of clusters is selected using inertia and cosine silhouette scores. In this final run, the best silhouette score among the tested values occurs at **k = 9**.

In [ ]:
elbow_scores

![TF-IDF elbow and silhouette plot](outputs/q2_elbow_tfidf.png)

In [ ]:
tfidf_summary

![TF-IDF cluster sizes](outputs/q2_cluster_sizes_tfidf.png)

**Cluster interpretation:** The top keywords and sample headlines give each cluster a meaningful topic label. TF-IDF clusters are highly interpretable because the most important centroid terms can be inspected directly.

## (g) Transformer-Based Embeddings and Comparison

The transformer stage uses `SentenceTransformer all-MiniLM-L6-v2` to encode each article into a dense semantic vector. Unlike TF-IDF, transformer embeddings can place documents close together when they have similar meaning even if they do not share many exact words.

In [ ]:
def transformer_or_svd_embeddings(clean_text, tfidf_matrix):
    try:
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer('all-MiniLM-L6-v2')
        embeddings = model.encode(clean_text, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
        return np.asarray(embeddings), 'SentenceTransformer all-MiniLM-L6-v2'
    except Exception as exc:
        svd = TruncatedSVD(n_components=100, random_state=RANDOM_STATE)
        embeddings = svd.fit_transform(tfidf_matrix)
        return embeddings, f'TruncatedSVD fallback: {type(exc).__name__}: {exc}'

# The final script run used real SentenceTransformer embeddings.
model_comparison

In [ ]:
embedding_summary

![Transformer cluster sizes](outputs/q2_cluster_sizes_embedding.png)

**Comparison examples:** The transformer clusters are more balanced and semantically coherent. For example, articles about politics can cluster together even when one headline uses `campaign`, another uses `president`, and another uses `GOP`. TF-IDF may miss this if exact vocabulary overlap is weak.

## (h) Evaluation and Insight

Since clustering is unsupervised, evaluation is qualitative and structural rather than simple accuracy. We inspect:

- top keywords per cluster,
- sample headlines per cluster,
- meaningful topic labels,
- cluster sizes,
- cosine silhouette scores,
- and comparison between TF-IDF and transformer embeddings.

In [ ]:
model_comparison

![Model comparison](outputs/q2_model_comparison.png)

### Final Answer

The transformer embedding approach produced the stronger clustering result in this run. The TF-IDF silhouette score was about **0.0069**, while the SentenceTransformer embedding silhouette score was about **0.0448**. Both values are low because short news snippets are broad and overlapping, but the transformer score is clearly higher.

**Strength of TF-IDF:** easy to interpret using top words from cluster centroids.  
**Weakness of TF-IDF:** depends heavily on exact word overlap.  
**Strength of transformer embeddings:** captures semantic similarity and context.  
**Weakness of transformer embeddings:** slower and more computationally expensive.

Overall, TF-IDF is useful for explanation, while transformer embeddings provide better semantic grouping.

## Extra Experiment: Testing Better Options

To make the work stronger, I tested more than one clustering setup instead of relying on a single TF-IDF K-Means run. This matters because clustering quality depends heavily on both the **text representation** and the **number of clusters**.

The additional options tested were:

- plain TF-IDF unigrams with K-Means,
- TF-IDF unigrams plus bigrams,
- stricter TF-IDF filtering using a lower `max_df`,
- MiniBatchKMeans as a faster K-Means variant,
- LSA/SVD-reduced TF-IDF, which compresses sparse TF-IDF into dense semantic dimensions,
- SentenceTransformer embeddings with a sweep over different `k` values.

The goal is not just to get the highest number, but to understand the trade-off between interpretability, speed, and semantic quality.

In [ ]:
options_comparison = pd.read_csv(OUTPUT_DIR / 'q2_better_options_comparison.csv')
transformer_k_sweep = pd.read_csv(OUTPUT_DIR / 'q2_transformer_k_sweep.csv')

options_comparison

![Clustering options tested](outputs/q2_better_options_comparison.png)

### What the Extra Tests Show

The strongest silhouette score came from **LSA/SVD over TF-IDF bigrams with 50 dimensions**. This is an important finding: reducing sparse TF-IDF into a smaller dense latent space can make K-Means work better because it removes noise and groups related terms into broader hidden dimensions.

The transformer model still performs much better than plain TF-IDF, but in this sample, a compact LSA representation produced the highest silhouette score. This does not mean LSA is always semantically better than transformers; it means that for K-Means and this specific 8,000-document sample, compressed TF-IDF created cleaner geometric separation.

## Transformer `k` Sweep

The original final clustering used **k = 9** so the TF-IDF and transformer results could be compared fairly with the same number of clusters. I also tested nearby values of `k` for transformer embeddings to see whether another cluster count separates the semantic vectors better.

In [ ]:
transformer_k_sweep

![Transformer k sweep](outputs/q2_transformer_k_sweep.png)

The transformer scores are close across `k = 5` to `k = 12`, which means there is no single obvious perfect number of clusters. In the tested range, **k = 11** gives the highest transformer silhouette score, while **k = 9** remains a good choice for direct comparison with the TF-IDF experiment.

For an exam presentation, this is a useful point: the selected `k` is defensible, but we also tested alternatives and can explain why the result is not arbitrary.

## 2D View of Transformer Embeddings

The plot below projects the 384-dimensional SentenceTransformer embeddings down to two dimensions using SVD. This visual is only an approximation, but it helps show that semantic document vectors form broad regions rather than perfectly separated islands.

![Transformer embedding projection](outputs/q2_transformer_embedding_projection.png)

## Improved Recommendation

Based on the additional testing, the best practical recommendation is:

| Need | Recommended option | Reason |
| --- | --- | --- |
| Simple explanation | TF-IDF + K-Means | Easy to explain with top keywords and centroids. |
| Better traditional clustering | TF-IDF bigrams + LSA/SVD + K-Means | Highest silhouette in the option test; reduces sparse-vector noise. |
| Better semantic grouping | SentenceTransformer + K-Means | Captures context and meaning beyond exact word overlap. |
| Fast large-scale production | MiniBatchKMeans | Faster on large datasets, though quality may drop slightly. |

For the final answer, I would present TF-IDF K-Means as the required baseline, SentenceTransformer K-Means as the semantic comparison, and mention that LSA/SVD was a strong additional improvement tested during experimentation.

## More Detailed Critical Analysis

### Why plain TF-IDF scores are low

News snippets are short and broad. Two articles can be about related topics while sharing only a few exact words. TF-IDF represents documents as sparse lexical vectors, so it struggles when semantic similarity exists without vocabulary overlap.

### Why LSA/SVD helps

SVD compresses thousands of TF-IDF features into fewer latent dimensions. This can merge related terms into shared hidden concepts and reduce the effect of noisy one-off words. That is why the 50-dimensional LSA option produced better cluster separation.

### Why transformers help

Transformers encode context. For example, articles mentioning `GOP`, `Republicans`, `White House`, and `president` can be placed near each other even if they do not repeat the exact same terms. This makes transformer embeddings more semantically meaningful than raw TF-IDF.

### Why silhouette is still modest

Even the better scores are not extremely high because news categories overlap naturally. Politics, media, world news, health, and social issues often share vocabulary and themes. Also, K-Means assumes roughly spherical clusters, which may not perfectly match real news-topic structure.

## Presentation Notes

When presenting this question, use this order:

1. Show the first 10 records and explain `headline`, `short_description`, and `category`.
2. Show preprocessing examples and explain why cleaning improves clustering.
3. Show top words/n-grams to prove there are visible topic patterns.
4. Show BoW vs TF-IDF matrix shapes and explain sparsity.
5. Show cosine similarity pairs and shared terms.
6. Show the elbow plot and justify `k = 9`.
7. Show TF-IDF cluster labels and keywords.
8. Show transformer cluster labels and the silhouette comparison.
9. Conclude that transformers perform better semantically, while TF-IDF is easier to interpret.

## Companion Files

The following files support this notebook:

- `q2_news_clustering.py`: complete reproducible script.
- `outputs/q2_report.md`: written report answer.
- `outputs/q2_tfidf_cluster_summary.csv`: TF-IDF cluster labels, keywords, and examples.
- `outputs/q2_embedding_cluster_summary.csv`: transformer cluster labels, terms, and examples.
- `outputs/q2_similarity_pairs.csv`: most similar TF-IDF document pairs.
- `outputs/q2_elbow_tfidf.png`: elbow/silhouette visual.
- `outputs/q2_model_comparison.csv`: TF-IDF vs transformer silhouette comparison.
- `outputs/q2_better_options_comparison.csv`: extra option-testing results.
- `outputs/q2_transformer_k_sweep.csv`: transformer `k` sweep results.
- `outputs/q2_better_options_comparison.png`: visual comparison of tested options.
- `outputs/q2_transformer_k_sweep.png`: transformer cluster-count comparison.
- `outputs/q2_transformer_embedding_projection.png`: 2D projection of transformer embeddings.